TASK 01

In [ ]:
import heapq

# Sample simplified "chess states"
# Each move leads to (next_state, score)
game_tree = {
    'start': [('A', 3), ('B', 5), ('C', 2)],
    'A': [('A1', 6), ('A2', 4)],
    'B': [('B1', 7), ('B2', 3)],
    'C': [('C1', 5)],
    'A1': [], 'A2': [],
    'B1': [], 'B2': [],
    'C1': []
}

def beam_search(start, depth_limit, beam_width):
    beam = [(0, [start])]  # (score, path)

    for _ in range(depth_limit):
        candidates = []

        for score, path in beam:
            node = path[-1]

            for neighbor, value in game_tree.get(node, []):
                new_score = score + value
                new_path = path + [neighbor]
                candidates.append((new_score, new_path))

        # Keep top-k highest score paths
        beam = heapq.nlargest(beam_width, candidates, key=lambda x: x[0])

    best = max(beam, key=lambda x: x[0])
    return best

path, score = beam_search('start', depth_limit=2, beam_width=2)
print("Best Move Sequence:", path)
print("Evaluation Score:", score)

Best Move Sequence: 12
Evaluation Score: ['start', 'B', 'B1']


TASK 02

In [ ]:
import random
import math

def distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def total_distance(route):
    dist = 0
    for i in range(len(route)-1):
        dist += distance(route[i], route[i+1])
    return dist

# Generate neighbor (swap two cities)
def get_neighbor(route):
    new_route = route[:]
    i, j = random.sample(range(len(route)), 2)
    new_route[i], new_route[j] = new_route[j], new_route[i]
    return new_route

def hill_climbing(points):
    current = points[:]
    random.shuffle(current)

    while True:
        neighbor = get_neighbor(current)

        if total_distance(neighbor) < total_distance(current):
            current = neighbor
        else:
            break

    return current, total_distance(current)

# Example coordinates
points = [(0,0), (2,3), (5,2), (6,6), (8,3)]

route, dist = hill_climbing(points)
print("Optimized Route:", route)
print("Total Distance:", dist)

Optimized Route: [(2, 3), (0, 0), (5, 2), (6, 6), (8, 3)]
Total Distance: 16.71937298368014


TASK 03

In [ ]:
import random
import math

# Generate cities
cities = [(random.randint(0,50), random.randint(0,50)) for _ in range(10)]

def distance(p1, p2):
    return math.sqrt((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)

def total_distance(route):
    return sum(distance(route[i], route[i+1]) for i in range(len(route)-1))

def create_route():
    route = cities[:]
    random.shuffle(route)
    return route

# Fitness (lower distance = better)
def fitness(route):
    return 1 / total_distance(route)

def select(population):
    population.sort(key=lambda x: total_distance(x))
    return population[:len(population)//2]

def crossover(p1, p2):
    point = random.randint(1, len(p1)-2)
    child = p1[:point]

    for city in p2:
        if city not in child:
            child.append(city)

    return child

def mutate(route):
    i, j = random.sample(range(len(route)), 2)
    route[i], route[j] = route[j], route[i]
    return route

def genetic_algorithm():
    population = [create_route() for _ in range(10)]

    for _ in range(100):
        population = select(population)
        new_pop = []

        while len(new_pop) < 10:
            p1, p2 = random.sample(population, 2)
            child = crossover(p1, p2)

            if random.random() < 0.1:
                child = mutate(child)

            new_pop.append(child)

        population = new_pop

    best = min(population, key=lambda x: total_distance(x))
    return best, total_distance(best)

best_route, best_dist = genetic_algorithm()
print("Best Route:", best_route)
print("Minimum Distance:", best_dist)

Best Route: [(40, 43), (48, 34), (38, 0), (22, 7), (11, 16), (12, 16), (38, 24), (39, 39), (11, 49), (6, 15)]
Minimum Distance: 186.49265991761584


TASK 04

In [ ]:
import heapq

# Jobs: (execution_time, priority)
jobs = [
    ('J1', 4, 2),
    ('J2', 3, 1),
    ('J3', 6, 3),
    ('J4', 2, 2)
]

processors = 2

# Evaluate load (max load among processors)
def evaluate(state):
    return max(state)

def beam_search(jobs, processors, beam_width):
    beam = [([0]*processors, [])]  # (loads, assignments)

    for job, time, priority in jobs:
        candidates = []

        for loads, assignment in beam:
            for p in range(processors):
                new_loads = loads[:]
                new_loads[p] += time

                new_assignment = assignment + [(job, p)]
                score = evaluate(new_loads)

                candidates.append((score, new_loads, new_assignment))

        # Select best k (min load)
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
        beam = [(x[1], x[2]) for x in beam]

    best = min(beam, key=lambda x: evaluate(x[0]))
    return best

loads, assignment = beam_search(jobs, processors, beam_width=2)

print("Task Allocation:", assignment)
print("Processor Loads:", loads)

Task Allocation: [('J1', 0), ('J2', 1), ('J3', 1), ('J4', 0)]
Processor Loads: [6, 9]
